In [ ]:
import sys
import os

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import optax

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_best_runs,
    load_top_n_runs,
    extract_history,
    print_best_hyperparameters,
    save_best_hyperparameters,
    get_optimizer_colors,
    ALL_OPTIMIZERS,
)
from optimizer_registry import create_optimizer, needs_loss

jax.config.update("jax_enable_x64", True)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


# Configuration


In [ ]:
# Backend: "wandb" or "local"
BACKEND = "local"

# WandB settings (ignored if BACKEND == "local")
PROJECT = "induced_metric"
ENTITY = "thomas_harvey"

# Results directory (for local backend)
RESULTS_DIR = os.path.join('..', 'results')

# Test functions to analyse
FUNCTIONS = ['beale', 'rosenbrock', 'himmelblau', 'ackley', 'rastrigin']

# Optimizers to analyse
OPTIMIZERS = ALL_OPTIMIZERS

# Iteration/batch number to load results from
ITERATION = 0

# Optimisation settings (for re-running best configs to get paths)
MAX_ITERATIONS = 1000

colors = get_optimizer_colors(OPTIMIZERS)
print(f"Backend: {BACKEND}")
print(f"Functions: {FUNCTIONS}")
print(f"Iteration: {ITERATION}")
print(f"Optimizers ({len(OPTIMIZERS)}): {OPTIMIZERS}")

# Test Functions


In [ ]:
@jax.jit
def beale_function(params):
    x, y = params
    return (1.5 - x + x * y)**2 + (2.25 - x + x * y**2)**2 + (2.625 - x + x * y**3)**2

@jax.jit
def rosenbrock_function(params):
    x, y = params
    return (1.0 - x)**2 + 100.0 * (y - x**2)**2

@jax.jit
def himmelblau_function(params):
    x, y = params
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

@jax.jit
def ackley_function(params):
    x, y = params
    a, b, c = 20.0, 0.2, 2 * jnp.pi
    return (-a * jnp.exp(-b * jnp.sqrt(0.5 * (x**2 + y**2)))
            - jnp.exp(0.5 * (jnp.cos(c * x) + jnp.cos(c * y)))
            + a + jnp.e)

@jax.jit
def rastrigin_function(params):
    x, y = params
    A = 10.0
    return A * 2 + (x**2 - A * jnp.cos(2 * jnp.pi * x)) + (y**2 - A * jnp.cos(2 * jnp.pi * y))

test_functions = {
    'beale': {
        'func': beale_function,
        'grad': jax.jit(jax.grad(beale_function)),
        'global_min': jnp.array([3.0, 0.5]),
        'initial': jnp.array([0.0, 0.0]),
        'xlim': (-1, 4), 'ylim': (-1, 1),
        'tolerance': 1e-7,
    },
    'rosenbrock': {
        'func': rosenbrock_function,
        'grad': jax.jit(jax.grad(rosenbrock_function)),
        'global_min': jnp.array([1.0, 1.0]),
        'initial': jnp.array([0.0, 0.0]),
        'xlim': (-0.5, 1.5), 'ylim': (-0.5, 2.0),
        'tolerance': 1e-7,
    },
    'himmelblau': {
        'func': himmelblau_function,
        'grad': jax.jit(jax.grad(himmelblau_function)),
        'global_min': jnp.array([3.0, 2.0]),
        'initial': jnp.array([0.0, 0.0]),
        'xlim': (-5, 5), 'ylim': (-5, 5),
        'tolerance': 1e-6,
    },
    'ackley': {
        'func': ackley_function,
        'grad': jax.jit(jax.grad(ackley_function)),
        'global_min': jnp.array([0.0, 0.0]),
        'initial': jnp.array([2.0, 2.0]),
        'xlim': (-5, 5), 'ylim': (-5, 5),
        'tolerance': 1e-6,
    },
    'rastrigin': {
        'func': rastrigin_function,
        'grad': jax.jit(jax.grad(rastrigin_function)),
        'global_min': jnp.array([0.0, 0.0]),
        'initial': jnp.array([2.0, 2.0]),
        'xlim': (-5, 5), 'ylim': (-5, 5),
        'tolerance': 1e-6,
    },
}

print("Test functions:")
for name, info in test_functions.items():
    print(f"  {name}: global minimum at {info['global_min']}")


# Load Results


In [ ]:
# Load best runs for each function
all_best_runs = {}
for func_name in FUNCTIONS:
    task_tag = f"small_examples_{func_name}"
    print(f"\n--- {func_name.upper()} ---")

    best_runs = load_best_runs(
        backend=BACKEND,
        optimizers=OPTIMIZERS,
        task_tag=task_tag,
        project=PROJECT,
        entity=ENTITY,
        results_dir=RESULTS_DIR,
        metric_key="sweep_metric",
        direction="minimize",
        sort_metric="sweep_metric",
        sort_order="+",
        iteration=ITERATION,
    )
    all_best_runs[func_name] = best_runs

print(f"\nLoaded results for {len(all_best_runs)} functions")
for func_name, runs in all_best_runs.items():
    print(f"  {func_name}: {len(runs)} optimizers")

# Performance Comparison


In [ ]:
# Build a summary DataFrame from all results
summary_data = []
for func_name, best_runs in all_best_runs.items():
    for opt, run_info in best_runs.items():
        s = run_info.get("summary", {})
        summary_data.append({
            'function': func_name,
            'optimizer': opt,
            'converged': s.get('converged', False),
            'iterations': s.get('iterations', MAX_ITERATIONS),
            'final_value': s.get('final_value', float('inf')),
            'runtime_seconds': s.get('runtime_seconds', 0),
            'runtime_ms': s.get('runtime_seconds', 0) * 1000,
        })

summary_df = pd.DataFrame(summary_data)
summary_df['optimizer'] = pd.Categorical(
    summary_df['optimizer'], categories=OPTIMIZERS, ordered=True
)
summary_df = summary_df.sort_values(['function', 'optimizer']).reset_index(drop=True)

print(f"Total results: {len(summary_df)}")
print(f"Convergence rate: {summary_df['converged'].mean():.1%}")

print("\nConvergence by optimizer:")
conv = summary_df.groupby('optimizer')['converged'].agg(['count', 'sum', 'mean'])
conv.columns = ['runs', 'converged', 'rate']
print(conv.to_string())


In [ ]:
# Performance comparison bar charts
import matplotlib.colors as mcolors

epsilon = 1e-6
converged_df = summary_df[summary_df['converged'] == True]

fig, axes = plt.subplots(2, 3, figsize=(32, 12))
fig.suptitle('Optimizer Performance Comparison', fontsize=16)
cmap = plt.cm.tab20

# 1. Iterations to convergence
ax = axes[0, 0]
if len(converged_df) > 0:
    pivot = converged_df.pivot(index='function', columns='optimizer', values='iterations')
    pivot.clip(lower=epsilon).plot(kind='bar', ax=ax, rot=45, colormap=cmap)
    ax.set_title('Iterations to Convergence')
    ax.set_ylabel('Iterations')
    ax.set_yscale('log')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

# 2. Runtime to convergence
ax = axes[0, 1]
if len(converged_df) > 0:
    pivot = converged_df.pivot(index='function', columns='optimizer', values='runtime_ms')
    pivot.clip(lower=epsilon).plot(kind='bar', ax=ax, rot=45, colormap=cmap)
    ax.set_title('Runtime to Convergence')
    ax.set_ylabel('Runtime (ms)')
    ax.set_yscale('log')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

# 3. Final values
ax = axes[0, 2]
pivot = summary_df.pivot(index='function', columns='optimizer', values='final_value')
pivot.clip(lower=epsilon).plot(kind='bar', ax=ax, rot=45, logy=True, colormap=cmap)
ax.set_title('Final Function Values (Log Scale)')
ax.set_ylabel('Function Value')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

# 4. Convergence heatmap
ax = axes[1, 0]
conv_matrix = summary_df.groupby(['optimizer', 'function'])['converged'].mean().unstack()
im = ax.imshow(conv_matrix.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(conv_matrix.columns)))
ax.set_xticklabels(conv_matrix.columns, rotation=45)
ax.set_yticks(range(len(conv_matrix.index)))
ax.set_yticklabels(conv_matrix.index, fontsize=7)
ax.set_title('Convergence Rate Heatmap')
plt.colorbar(im, ax=ax)
for i in range(len(conv_matrix.index)):
    for j in range(len(conv_matrix.columns)):
        ax.text(j, i, f'{conv_matrix.iloc[i, j]:.0%}',
                ha='center', va='center', fontsize=7)

# 5. Efficiency scatter
ax = axes[1, 1]
if len(converged_df) > 0:
    for i, opt in enumerate(converged_df['optimizer'].unique()):
        opt_data = converged_df[converged_df['optimizer'] == opt]
        ax.scatter(opt_data['iterations'], opt_data['runtime_ms'],
                   label=opt, alpha=0.7, s=80)
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Runtime (ms)')
    ax.set_title('Efficiency: Iterations vs Runtime')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

# 6. Rankings
ax = axes[1, 2]
if len(converged_df) > 0:
    avg_iter = converged_df.groupby('optimizer')['iterations'].mean().sort_values()
    bars = ax.barh(range(len(avg_iter)), avg_iter.values)
    ax.set_yticks(range(len(avg_iter)))
    ax.set_yticklabels(avg_iter.index, fontsize=8)
    ax.set_xlabel('Average Iterations')
    ax.set_title('Average Iterations to Convergence')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Convergence Curves


In [ ]:
# Plot convergence curves for each function using history data
for func_name in FUNCTIONS:
    best_runs = all_best_runs[func_name]
    if not best_runs:
        continue

    task_tag = f"small_examples_{func_name}"
    history_data = extract_history(
        BACKEND, best_runs, ["function_value", "runtime_seconds"]
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f'Convergence: {func_name.title()} Function', fontsize=14)

    tab_colors = plt.cm.tab20(np.linspace(0, 1, len(history_data)))

    for (opt, data), color in zip(history_data.items(), tab_colors):
        fv = data.get("function_value", [])
        iters = np.arange(len(fv))
        summary = best_runs[opt].get("summary", {})
        converged = summary.get("converged", False)
        n_iter = summary.get("iterations", len(fv))

        if len(fv) == 0:
            continue

        # vs iterations
        axes[0].semilogy(iters, fv, label=f"{opt} ({n_iter} iter)",
                         color=color, linewidth=2, alpha=0.8)
        if converged:
            idx = min(n_iter, len(fv) - 1)
            axes[0].scatter(idx, fv[idx], color=color, s=100, marker='*',
                            edgecolor='black', zorder=10)

        # vs runtime
        rt = data.get("runtime_seconds", [])
        if len(rt) > 0:
            rt_ms = np.array(rt) * 1000
        else:
            runtime_ms = summary.get("runtime_seconds", 0) * 1000
            rt_ms = np.linspace(0, runtime_ms, len(fv))

        axes[1].semilogy(rt_ms, fv[:len(rt_ms)],
                         label=f"{opt} ({rt_ms[-1]:.1f} ms)" if len(rt_ms) > 0 else opt,
                         color=color, linewidth=2, alpha=0.8)
        if converged and len(rt_ms) > 0:
            idx = min(n_iter, len(rt_ms) - 1)
            axes[1].scatter(rt_ms[idx], fv[idx], color=color, s=100, marker='*',
                            edgecolor='black', zorder=10)

    axes[0].set_xlabel('Iterations')
    axes[0].set_ylabel('Function Value (Log)')
    axes[0].set_title('vs Iterations')
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel('Runtime (ms)')
    axes[1].set_ylabel('Function Value (Log)')
    axes[1].set_title('vs Runtime')
    axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary
    print(f"\n{func_name.title()} Function Summary:")
    print("-" * 60)
    for opt in OPTIMIZERS:
        if opt not in best_runs:
            continue
        s = best_runs[opt].get("summary", {})
        status = "Converged" if s.get("converged", False) else "Failed"
        print(f"  {opt:25s}: {status:10s} | {s.get('iterations', 0):4d} iter | "
              f"{s.get('runtime_seconds', 0)*1000:6.1f} ms | Final: {s.get('final_value', float('inf')):.2e}")


# Rankings


In [ ]:
print("=" * 80)
print("OVERALL RANKINGS")
print("=" * 80)

print("\n1. BY CONVERGENCE RATE:")
conv_rate = summary_df.groupby('optimizer')['converged'].mean().sort_values(ascending=False)
for i, (opt, rate) in enumerate(conv_rate.items(), 1):
    print(f"  {i:2d}. {opt:25s}: {rate:.1%}")

converged_only = summary_df[summary_df['converged'] == True]

if len(converged_only) > 0:
    print("\n2. BY AVERAGE ITERATIONS (converged runs):")
    avg_iter = converged_only.groupby('optimizer')['iterations'].mean().sort_values()
    for i, (opt, val) in enumerate(avg_iter.items(), 1):
        print(f"  {i:2d}. {opt:25s}: {val:.1f}")

    print("\n3. BY AVERAGE RUNTIME (converged runs):")
    avg_rt = converged_only.groupby('optimizer')['runtime_ms'].mean().sort_values()
    for i, (opt, val) in enumerate(avg_rt.items(), 1):
        print(f"  {i:2d}. {opt:25s}: {val:.1f} ms")


# Best Hyperparameters


In [ ]:
# Print and save best hyperparameters per function
all_hp_rows = []

for func_name in FUNCTIONS:
    best_runs = all_best_runs[func_name]
    if not best_runs:
        continue

    print(f"\n{'=' * 60}")
    print(f"{func_name.upper()} FUNCTION")
    print(f"{'=' * 60}")
    print_best_hyperparameters(best_runs)

    for opt, run_info in best_runs.items():
        row = {
            'function': func_name,
            'optimizer': opt,
            'run_name': run_info.get('name', 'unknown'),
        }
        row.update(run_info.get('config', {}))
        row.update(run_info.get('summary', {}))
        all_hp_rows.append(row)

df = pd.DataFrame(all_hp_rows)
filename = f'best_hyperparameters_SmallExamples_itr_{ITERATION}.csv'
df.to_csv(filename, index=False)
print(f"\nSaved all hyperparameters to {filename}")
df